<a href="https://colab.research.google.com/github/juliajohanson/Sprint-01-PAI/blob/main/Sprint_01_PAI_05_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess, time

# 1. Instalar dependência zstd (necessária para o instalador do Ollama)
!apt-get install -y zstd -q

# 2. Instalar o servidor Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Iniciar o servidor em background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Aguardar o servidor subir (importante!)
time.sleep(5)
print("🟢 Servidor Ollama iniciado!")

# 4. Instalar o SDK Python
!pip install ollama -q

# 5. Baixar o modelo base (só na primeira vez, ~800MB)
!ollama pull llama3.2:1b

# 6. Verificar
import ollama
print("✅ Tudo pronto! Servidor rodando e modelo baixado.")

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (6,875 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama 

In [ ]:
import ollama

# Llama 3.2 de 1B
MODELO_BASE = "llama3.2:1b"

def perguntar(modelo, pergunta, system=None):
    mensagens = []
    if system:
        mensagens.append({"role": "system", "content": system})
    mensagens.append({"role": "user", "content": pergunta})

    resposta = ollama.chat(model=modelo, messages=mensagens)
    return resposta["message"]["content"]


In [ ]:
# Dataset: Assistente do ChargeGrid Goodwe
dataset = [

  {
    "pergunta": "O que é o sistema ChargeGrid Intelligence nos eletropostos GoodWe?",
    "resposta": "O ChargeGrid Intelligence é o cérebro dos eletropostos, integrando mecanismos de orquestração de potência, registro de ciclos de carga, sistemas de faturamento e protocolos de comunicação para otimizar a operação da estação."
  },
  {
    "pergunta": "Como funciona a orquestração de potência em locais com múltiplos carregadores?",
    "resposta": "A orquestração de potência utiliza o ChargeGrid Intelligence para balancear a carga disponível. Se vários veículos carregam simultaneamente, o sistema distribui a energia de forma inteligente para evitar sobrecarga na rede elétrica local."
  },
  {
    "pergunta": "O sistema EV ChargeOps é indicado para qual tipo de cenário?",
    "resposta": "O EV ChargeOps é ideal para o gerenciamento de uso compartilhado em condomínios residenciais ou comerciais, permitindo o controle de acesso e a individualização do consumo de energia."
  },
  {
    "pergunta": "Como é feito o faturamento das recargas realizadas em um eletroposto GoodWe?",
    "resposta": "Através do ChargeGrid Intelligence, o sistema registra o consumo exato de cada sessão. Esses dados são processados para gerar cobranças automáticas ou relatórios de faturamento detalhados para os usuários."
  },
  {
    "pergunta": "É possível monitorar a saúde da bateria através do registro de ciclos?",
    "resposta": "Sim, o sistema registra cada ciclo de carga completo e parcial. Esses dados fornecem um histórico preciso que auxilia na análise do desempenho e na preservação da vida útil das baterias dos veículos atendidos."
  },
  {
    "pergunta": "Como o EV ChargeOps gerencia quem pode usar o carregador no condomínio?",
    "resposta": "O gerenciamento é feito via autenticação (como cartões RFID ou aplicativos). O administrador do condomínio define permissões de acesso, garantindo que apenas usuários autorizados utilizem as estações."
  },
  {
    "pergunta": "Quais protocolos de comunicação o ChargeGrid Intelligence utiliza?",
    "resposta": "O sistema opera com protocolos padrão de mercado, como o OCPP (Open Charge Point Protocol), permitindo a comunicação entre o hardware do eletroposto e softwares de gestão de terceiros ou redes de recarga."
  },
  {
    "pergunta": "O sistema permite a divisão de custos de energia em áreas comuns?",
    "resposta": "Sim. O EV ChargeOps identifica o consumo por usuário ou unidade habitacional, facilitando o rateio preciso na conta de luz do condomínio ou a cobrança direta via plataforma de gestão."
  },
  {
    "pergunta": "O ChargeGrid Intelligence ajuda na proteção contra surtos de energia?",
    "resposta": "Além do monitoramento de software, o sistema coordena os mecanismos de proteção física, interrompendo a carga imediatamente caso detecte instabilidades na rede ou falhas de isolamento no veículo."
  },
  {
    "pergunta": "Consigo visualizar os dados de recarga em tempo real?",
    "resposta": "Sim, tanto o ChargeGrid Intelligence quanto o EV ChargeOps fornecem interfaces de monitoramento em tempo real, onde é possível verificar potência instantânea, tempo restante e custos acumulados."
  }
]

print(f"✅ Dataset criado com {len(dataset)} exemplos")

✅ Dataset criado com 10 exemplos


In [ ]:
def gerar_modelfile(modelo_base, system_prompt, exemplos):
    """Gera o conteúdo de um Modelfile para o Ollama."""
    linhas = []

    # 1. Modelo base
    linhas.append(f"FROM {modelo_base}\n")

    # 2. Parâmetros de geração
    linhas.append("PARAMETER temperature 0.5")
    linhas.append("PARAMETER num_predict 300\n")

    # 3. System prompt permanente
    linhas.append(f'SYSTEM """\n{system_prompt}\n"""\n')

    # 4. Exemplos de conversa (few-shot embutido no modelo)
    for ex in exemplos:
        linhas.append(f'MESSAGE user "{ex["pergunta"]}"')
        linhas.append(f'MESSAGE assistant "{ex["resposta"]}"\n')

    return "\n".join(linhas)
SYSTEM_PROMPT = """
Você é um assistente que auxilia clientes com duvidas sobre VE's e ChargeGrid da Goodwe.

Suas regras:
- Sempre responda em português brasileiro
- Seja amigável e objetivo
- Nunca prometa algo que não possa cumprir
- Foque em responder a duvida do cliente rapidamente e objetivamente
- Responda as perguntas fielmente de acordo com os dados do contexto

REGRAS CRÍTICAS:

1. Caso a pergunta esteja fora do escopo do contexto do ChargeGrid e da Goodwe retorne uma mensagem dizendo que não pode ajudar com a pergunta

"""

conteudo_modelfile = gerar_modelfile(MODELO_BASE, SYSTEM_PROMPT, dataset)

print("📄 Modelfile gerado:\n")
print(conteudo_modelfile)

📄 Modelfile gerado:

FROM llama3.2:1b

PARAMETER temperature 0.5
PARAMETER num_predict 300

SYSTEM """

Você é um assistente que auxilia clientes com duvidas sobre VE's e ChargeGrid da Goodwe.

Suas regras:
- Sempre responda em português brasileiro
- Seja amigável e objetivo
- Nunca prometa algo que não possa cumprir
- Foque em responder a duvida do cliente rapidamente e objetivamente
- Responda as perguntas fielmente de acordo com os dados do contexto

REGRAS CRÍTICAS:

1. Caso a pergunta esteja fora do escopo do contexto do ChargeGrid e da Goodwe retorne uma mensagem dizendo que não pode ajudar com a pergunta


"""

MESSAGE user "O que é o sistema ChargeGrid Intelligence nos eletropostos GoodWe?"
MESSAGE assistant "O ChargeGrid Intelligence é o cérebro dos eletropostos, integrando mecanismos de orquestração de potência, registro de ciclos de carga, sistemas de faturamento e protocolos de comunicação para otimizar a operação da estação."

MESSAGE user "Como funciona a orquestração de 

In [28]:
# Salvar o Modelfile em disco
with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(conteudo_modelfile)

# Criar o modelo customizado via CLI do Ollama
# O nome do nosso modelo personalizado será "Assistente ChargeGrid"
!ollama create "AssistenteChargeGrid" -f Modelfile

print("✅ Modelo customizado 'AssistenteChargeGrid' criado!")


✅ Modelo customizado 'AssistenteChargeGrid' criado!


In [29]:
MODELO_CUSTOM = "AssistenteChargeGrid"

In [31]:
print("=" * 60)
print("Teste modelo customizado")
print("=" * 60)

pergunta = "O ChargeGrid Intelligence permite priorizar a recarga do veículo utilizando apenas o excedente de energia solar?"
print(f"\n📌 Pergunta: {pergunta}\n")

# Antes de tentar usar o modelo customizado, vamos verificar se ele existe
print("Verificando modelos Ollama instalados:")
!ollama list

print("\n── Modelo CUSTOMIZADO ───────────────")
print(perguntar(MODELO_CUSTOM, pergunta))

Teste modelo customizado

📌 Pergunta: O ChargeGrid Intelligence permite priorizar a recarga do veículo utilizando apenas o excedente de energia solar?

Verificando modelos Ollama instalados:
NAME                           ID              SIZE      MODIFIED           
AssistenteChargeGrid:latest    ee6a0f1d2e45    1.3 GB    About a minute ago    
llama3.2:1b                    baf6a787fdff    1.3 GB    44 minutes ago        

── Modelo CUSTOMIZADO ───────────────
Sim, o sistema utiliza um mecanismo de priorização de carga que analisa os níveis de energia disponível no veículo e na rede elétrica para determinar a melhor opção de recarga. Isso inclui considerar o tempo restante de carga do veículo, a disponibilidade de energia solar e a demanda por recarga em diferentes horários.
